**MATH 170: Calculus for Data Science**


# **Lab 1 (LD):Perceptrons, Derivatives, and Activation Functions**

**Big Idea:** A perceptron with one input is just a function with one parameter:

$$
\hat{y}=\phi(wx)
$$

- $x$: input feature  
- $w$: weight (the only parameter today)  
- $\phi$: activation function  

Training = choosing $w$ so that a **loss function** $E(w)$ is as small as possible.

---

## How to work through this lab (self-guided)
In each section you’ll follow this routine:

1) **Try a few values of $w$** (data science style)  
2) **Compute the derivative $E'(w)$** (calculus style)  
3) Use the **increasing–decreasing test** + **critical numbers** to explain what’s happening  

> When you see **GTW**, you should write code or write a short response.

---

## Learning Targets (VARIOSA)
- **V:** plot models + loss curves + activation functions  
- **A:** approximate “best” $w$ using tables / grids  
- **I/O:** tune parameters by minimizing $E(w)$  
- **S:** compute derivatives with SymPy  
- **R:** interpret logistic outputs as probabilities for classification


In [ ]:
# Part 0 — Setup (run this first)
import numpy as np
import sympy as sp
import pandas as pd
import plotly.express as px

sp.init_printing()

# Symbols
x = sp.Symbol('x', real=True)
w = sp.Symbol('w', real=True)

print("Ready!")



---

# Part 1 — Regression Perceptron (Identity Activation)

## Context: Energy Drinks Consumed Before Noon vs Energy at 2PM
We track:
- $x$ = Energy drinks consumed before noon  
- $y$ = energy level at 2PM (1–10 scale)

We fit a simple perceptron regression model (identity activation):

$$
\hat{y} = wx
$$

### Dataset ($n=5$)


In [ ]:
# Data (n=5)
x_data = np.array([1, 2, 3, 4, 5], dtype=float)
y_data = np.array([3, 6, 4, 8, 10], dtype=float)

df_reg = pd.DataFrame({"x": x_data, "y": y_data})
df_reg



### 1.1 Visualize the data
**GTW (thought question):** Does it look like energy increases or decreases energy drinks consumed before noon increase?


In [ ]:
fig = px.scatter(df_reg, x="x", y="y", title="TikTok hours (x) vs Energy at 2PM (y)")
fig.show()



## 1.2 Build the loss function $E(w)$

Model:
$$
\hat{y}_i = wx_i
$$

Least squares loss:
$$
E(w)=\sum_{i=1}^{n}(y_i-wx_i)^2
$$

**GTW:** Read the code and make sure you understand what each line is doing. Write out by hand the five terms in the loss function sum.


First five terms are ...

In [ ]:
# Build E(w) symbolically with SymPy
E = 0
for xi, yi in zip(x_data, y_data):
    E += (w*xi - yi)**2

E = sp.simplify(E)
E



## 1.3 Try different values of $w$ AND check the derivative

When you “try weights” you’re really probing whether the loss is increasing or decreasing.

- If $E'(w) < 0$, then $E(w)$ is **decreasing** at that $w$.
- If $E'(w) > 0$, then $E(w)$ is **increasing** at that $w$.

**GTW:** Find the derivative $E'(w)$. For what values of $w$ is it positive? Negative? How do these values relate to the behavior of $E(w)$?


$E'(w)$ is positive when $w$ is ...

$E'(w)$ is negative when $w$ is ...

In [ ]:
# Compute the derivative
E_prime = sp.simplify(sp.diff(E, w))
...


In [ ]:
# Numeric versions for quick evaluation
E_num = sp.lambdify(w, E, "numpy")
Eprime_num = sp.lambdify(w, E_prime, "numpy")

# GTW: Try these weights (you can add more)
w_try = np.array([-2, -1, 0, 1, 2, 3], dtype=float)

table = pd.DataFrame({
    "w": w_try,
    "E(w)": E_num(w_try),
    "E'(w)": Eprime_num(w_try)
})

# Add an "increasing/decreasing" interpretation
table["Loss Behavior near w"] = np.where(table["E'(w)"] > 0, "increasing", np.where(table["E'(w)"] < 0, "decreasing", "flat (critical)"))
table


**GTW:** Based on the above, where do you think the minimum loss occurs?

The minimum occurs ...


## 1.4 Loss curve + where the minimum happens

**GTW:**
1. Plot $E(w)$ for a wide range of $w$.
2. Where does the minimum *look* like it occurs?


In [ ]:
w_vals = np.linspace(-5, 5, 600)
loss_vals = E_num(w_vals)

df_loss = pd.DataFrame({"w": w_vals, "E(w)": loss_vals})
fig = px.line(df_loss, x="w", y="E(w)", title="Loss curve E(w) for regression perceptron")
fig.show()



## 1.5 Critical number and First Derivative Test (calculus)

A **critical number** occurs where:

$$
E'(w)=0 \quad \text{or} \quad E'(w) \text{ does not exist}
$$

Here $E'(w)$ exists everywhere (it’s a polynomial), so we solve:

$$
E'(c)=0
$$

**GTW:** Solve for the critical number $c$. Then explain why it should be a minimum using an increasing/decreasing argument.


The critical number is ...

In [ ]:
w_star = sp.solve(sp.Eq(E_prime, 0), w)
...


Let's visualize the data with the line of best fit.

In [ ]:
# Plot data + best-fit model yhat = w_best * x
xs = np.linspace(0.5, 5.5, 200)
yhat = w_best * xs

fig = px.scatter(df_reg, x="x", y="y", title=f"Best regression perceptron: yhat = {w_best:.3f} x")
fig = fig.add_scatter(x=xs, y=yhat, mode="lines", name="model")
fig.show()



**GTW (short response):**  
At the best weight $c$, what is $E'(c)$?  
What does that tell you about the slope of the loss curve at the minimum?


In [ ]:
Eprime_num(w_best)



---

# Part 2 — Binary Classification (Same Architecture, different $\phi$)

Now outputs are classes:
- 0 = not viral
- 1 = viral

We still use the score:
$$
z = wx
$$

but we change the activation function.

---

## Part 2A — Heaviside Step Activation (hard threshold)

$$
H(z)=
\begin{cases}
0 & z<0\\
1 & z\ge 0
\end{cases}
$$

Model:
$$
\hat{y}=H(wx)
$$

### Context: “Viral or not?”
- $x$ = engagement score (negative means weak engagement, positive means strong)


In [ ]:
# Classification dataset (n=5)
x_cls = np.array([-2, -1, 0.5, 2, 3], dtype=float)
y_cls = np.array([0, 1, 0, 1, 1], dtype=int)

df_cls = pd.DataFrame({"x": x_cls, "y": y_cls})
df_cls



### 2A.1 Try a few weights and count misclassifications

**GTW:**
1. Try $w = 0.5, 1, 2, 3$.
2. For each $w$, compute predictions $\hat{y}$.
3. Count how many are wrong.

> Notice: With Heaviside, the model output can “jump” suddenly.


In [ ]:
def heaviside(z):
    return (z >= 0).astype(int)

def misclassifications(y_true, y_pred):
    return int(np.sum(y_true != y_pred))

w_try_step = np.array([-2, -1, 0, 0.5, 1, 2, 3], dtype=float)

rows = []
for wv in w_try_step:
    yhat = heaviside(wv * x_cls)
    rows.append({
        "w": wv,
        "predictions": list(yhat),
        "misclassifications": misclassifications(y_cls, yhat)
    })

pd.DataFrame(rows)



### 2A.2 Why calculus struggles here
The step function is **not differentiable** at 0, so it doesn’t give us a useful derivative-based “direction” for improving $w$.

**GTW (short response):**  
In your own words, why is “try-and-check” basically the only option here?



---

## Part 2B — Logistic Activation (soft threshold)

$$
\sigma(z)=\frac{1}{1+e^{-z}}
$$

Model:
$$
\hat{y}=\sigma(wx)
$$

Now $\hat{y}$ is a number between 0 and 1, which we interpret as a **probability of class 1**.

To convert probability to a class label, a common rule is:
- predict 1 if $\hat{y}\ge 0.5$
- predict 0 otherwise

### Loss function for training (squared error)
$$
E(w)=\sum_{i=1}^{n}\left(\sigma(wx_i)-y_i\right)^2
$$

Key difference: **This loss is differentiable**, so we can study $E'(w)$.


In [ ]:
# Build logistic predictions symbolically (as a function of x, w)
sigma_sym = 1/(1 + sp.exp(-(w*x)))
sigma_sym


In [ ]:
# Build E(w) for logistic classification (squared error)
E_log = 0
for xi, yi in zip(x_cls, y_cls):
    E_log += ( (1/(1 + sp.exp(-(w*xi)))) - yi )**2

E_log = sp.simplify(E_log)
E_log


In [ ]:
# Derivative of logistic loss
E_log_prime = sp.simplify(sp.diff(E_log, w))
E_log_prime



### 2B.1 Try different $w$ values AND check $E'(w)$ each time

**GTW:**
Try:
$$
w=0.5,\ 1,\ 2,\ 3,\ 4
$$

For each $w$:
- compute $E(w)$
- compute $E'(w)$
- decide if the loss is increasing or decreasing there


In [ ]:
Elog_num = sp.lambdify(w, E_log, "numpy")
Elogprime_num = sp.lambdify(w, E_log_prime, "numpy")

w_try_log = np.array([-2, -1, 0, 0.5, 1, 2, 3, 4], dtype=float)

tab_log = pd.DataFrame({
    "w": w_try_log,
    "E(w)": Elog_num(w_try_log),
    "E'(w)": Elogprime_num(w_try_log),
})
tab_log["Behavior near w"] = np.where(tab_log["E'(w)"] > 0, "increasing", np.where(tab_log["E'(w)"] < 0, "decreasing", "flat (critical)"))
tab_log



### 2B.2 Visualize: predictions for different weights

**GTW:** What happens to the “sharpness” of the probability curve as $w$ increases?


In [ ]:
def logistic(z):
    return 1/(1+np.exp(-z))

xs = np.linspace(-4, 4, 600)

curves = []
for wv in [0.5, 1, 2, 4]:
    curves.append(pd.DataFrame({
        "x": xs,
        "yhat": logistic(wv*xs),
        "w": f"w={wv}"
    }))

df_curves = pd.concat(curves, ignore_index=True)
fig = px.line(df_curves, x="x", y="yhat", color="w",
              title="Logistic activation outputs: yhat = sigma(w x)")
fig.show()



### 2B.3 Loss curve + (approx) critical number

**GTW:**
1. Plot $E(w)$ vs $w$ for $w\in[-2,6]$.
2. Where is the minimum?
3. Use the sign of $E'(w)$ to explain why the minimum should be near that point.


In [ ]:
w_vals = np.linspace(-2, 6, 800)
df_elog = pd.DataFrame({"w": w_vals, "E(w)": Elog_num(w_vals)})

fig = px.line(df_elog, x="w", y="E(w)", title="Logistic classification loss E(w)")
fig.show()


In [ ]:
# Approximate the minimizer by grid search (data science style)
w_grid = np.linspace(-2, 6, 4001)
E_grid = Elog_num(w_grid)
idx = np.argmin(E_grid)
w_best_log = float(w_grid[idx])
E_best_log = float(E_grid[idx])

w_best_log, E_best_log


In [ ]:
# Check derivative at the best weight (should be near 0)
Elogprime_num(w_best_log)



**GTW (short response):**
- If $E'(w)$ is negative at your current $w$, should you increase or decrease $w$?
- If $E'(w)$ is positive, should you increase or decrease $w$?
Explain using the increasing–decreasing test.



---

# Part 3 — Activation Functions as Functions (calculus properties)

Now we ignore neural nets for a moment and analyze activation functions like “essential functions”.

We’ll study:
1) Logistic $\sigma(x)$  
2) Hyperbolic tangent $\tanh(x)$

We want:
- asymptotes (limits at infinity)
- derivative (slope)
- increasing/decreasing
- max slope (where is the function most sensitive?)


In [ ]:
# Define logistic and tanh as functions of x (not w)
sigma_x = 1/(1 + sp.exp(-x))
tanh_x = ...

sigma_x, tanh_x



## 3.1 Logistic: asymptotes

**GTW:** Compute:
$$
\lim_{x\to\infty}\sigma(x),\quad \lim_{x\to-\infty}\sigma(x)
$$


In [ ]:
sp.limit(sigma_x, x, sp.oo), sp.limit(sigma_x, x, -sp.oo)



## 3.2 Logistic: derivative, increasing/decreasing, and max slope

**GTW:**
1. Compute $\sigma'(x)$.  
2. Is it ever negative? What does that tell you?  
3. Evaluate $\sigma'(0)$.  
4. Find where $\sigma'(x)$ is largest (hint: solve $\sigma''(x)=0$).


In [ ]:
sigma_prime_x = sp.diff(sigma_x, x)
sigma_prime_x


In [ ]:
# Check sigma'(0)
sigma_prime_x.subs(x, 0)


In [ ]:
# Find candidate for max slope by solving sigma''(x)=0
sigma_second_x = sp.simplify(sp.diff(sigma_prime_x, x))
sp.solve(sp.Eq(sigma_second_x, 0), x)



## 3.3 Tanh: asymptotes

**GTW:** Compute:
$$
\lim_{x\to\infty}\tanh(x),\quad \lim_{x\to-\infty}\tanh(x)
$$


In [ ]:
sp.limit(tanh_x, x, sp.oo), sp.limit(tanh_x, x, -sp.oo)



## 3.4 Tanh: derivative, increasing/decreasing, and max slope

**GTW:**
1. Compute $\tanh'(x)$.  
2. Evaluate $\tanh'(0)$.  
3. Where is the slope largest?  
4. Compare logistic vs tanh: which has bigger slope at 0?


In [ ]:
tanh_prime = sp.diff(tanh_x, x)
tanh_prime


In [ ]:
sp.simplify(tanh_prime.subs(x, 0))


In [ ]:
# Candidate for max slope of tanh by solving tanh''(x)=0
tanh_second = sp.simplify(sp.diff(tanh_prime, x))
sp.solve(sp.Eq(tanh_second, 0), x)



## 3.5 Plot logistic and tanh

**GTW:** On the plot, identify:
- the horizontal asymptotes for each
- the “most sensitive” region (largest slope)


In [ ]:
xs = np.linspace(-6, 6, 800)
sigma_np = sp.lambdify(x, sigma_x, "numpy")(xs)
tanh_np = sp.lambdify(x, tanh_x, "numpy")(xs)

df_act = pd.DataFrame({"x": xs, "logistic": sigma_np, "tanh": tanh_np})
df_long = df_act.melt(id_vars="x", var_name="function", value_name="y")

fig = px.line(df_long, x="x", y="y", color="function", title="Activation functions: logistic vs tanh")
fig.show()



---

# Suggested Homework (submit on Canvas)

### HW1 — New regression perceptron (identity activation)
Use the dataset:

| $x$ | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|
| $y$ | 1 | 2 | 4 | 4 | 7 |

1. Build $E(w)=\sum(wx_i-y_i)^2$.  
2. Compute $E'(w)$.  
3. Try at least 6 weights and make a table of $w, E(w), E'(w)$.  
4. Find the critical number $w^\*$ and explain (using increasing/decreasing) why it’s a minimum.

---

### HW2 — Logistic classification derivative check
Use the dataset:

$$
x=[-3,-2,-1,1,2],\quad y=[0,0,0,1,1]
$$

Let:
$$
E(w)=\sum(\sigma(wx_i)-y_i)^2
$$

1. Try $w=0.5,1,2,3,4$ and compute $E(w)$ and $E'(w)$ each time.  
2. Based on the sign of $E'(w)$, explain which direction you would move $w$ to reduce loss.

---

### HW3 — Activation function comparison (words + math)
1. State the horizontal asymptotes of logistic and tanh.  
2. Compute $\sigma'(0)$ and $\tanh'(0)$.  
3. In 3–5 sentences: why is “big slope near 0” useful for learning/classification?

---

